# Análisis de Precios de Carburantes con GenAI

## Un ejercicio de divulgación: cómo usar IA como copiloto en análisis de datos

Aprenderemos a analizar precios de carburantes españoles usando IA para acelerar cada fase: descarga, limpieza, exploración, ingeniería de variables y modelado.

---
## FASE 0: Preparación del Entorno

Configurar metadatos, importar librerías y verificar que todo está listo.

In [ ]:
# ========================================
# METADATOS DEL NOTEBOOK
# ========================================
VERSION_NOTEBOOK = "0.2.0"
NUMERO_ITERACION = 5

print("=" * 70)
print(f" ANÁLISIS DE CARBURANTES CON GENAI")
print(f" Versión: v{VERSION_NOTEBOOK} | Iteración: {NUMERO_ITERACION}")
print("=" * 70)
print()

# ========================================
# FASE 0: IMPORTAR LIBRERÍAS REQUERIDAS
# ========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualizaciones
plt.style.use('default')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(" Librerías importadas correctamente")
print(f" - pandas {pd.__version__}")
print(f" - numpy {np.__version__}")
print(f" - plotly para gráficos interactivos")

In [ ]:
# ========================================
# IMPORTACIONES Y CONFIGURACIÓN
# ========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
import json
import subprocess

# Imports para descarga robusta de APIs
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Configuración
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

print(f" Pandas {pd.__version__} | NumPy {np.__version__} | Plotly (interactivo)")
print(f" Entorno preparado con visualizaciones interactivas")

---
## FASE 1: Ingesta de Datos

Descargar dataset de precios de carburantes desde la API del Ministerio de Turismo.

In [ ]:
def descargar_datos_api(url):
 """
 Descarga datos con fallback robusto: requests → curl (insecure) → estructura vacía.
 Retorna un diccionario JSON-compatible.
 """
 # Intento 1: requests con verify=False para evitar SSL
 try:
 print(" Intentando con requests...")
 sesion = requests.Session()
 sesion.headers.update({
 "Accept": "application/json",
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
 })
 reintentos = Retry(total=2, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
 sesion.mount("https://", HTTPAdapter(max_retries=reintentos))

 response = sesion.get(url, timeout=45, verify=False)
 response.raise_for_status()
 print(" OK con requests")
 return response.json()
 except Exception as e:
 print(f" requests falló: {type(e).__name__}")

 # Intento 2: curl con -k (insecure)
 try:
 print(" Intentando con curl (insecure)...")
 resultado = subprocess.run(
 ["curl", "-s", "-L", "-k", "--max-time", "60",
 "-H", "Accept: application/json",
 "-H", "User-Agent: Mozilla/5.0",
 url],
 capture_output=True, text=True, timeout=75
 )
 if resultado.returncode == 0 and resultado.stdout.strip():
 datos = json.loads(resultado.stdout)
 print(" OK con curl")
 return datos
 else:
 print(f" curl retornó código {resultado.returncode}")
 except Exception as e:
 print(f" curl falló: {type(e).__name__}")

 # Fallback: estructura estándar vacía
 print(" ⚠ API no disponible. Datos demo serán usados.")
 return {
 "Fecha": "N/A",
 "ResultadoConsulta": "SIN_CONEXION",
 "ListaEESSPrecio": []
 }

In [ ]:
# T009-T010: Descargar y cargar datos
print("\n[T009-T010] Descargando e ingestando dataset...")
print("=" * 70)

url_api = "https://sedeaplicaciones.minetur.gob.es/ServiciosRESTCarburantes/PreciosCarburantes/EstacionesTerrestres/"

try:
 datos_json = descargar_datos_api(url_api)
 
 # Cargar en DataFrame
 df_raw = pd.DataFrame(datos_json.get('ListaEESSPrecio', []))
 
 if len(df_raw) == 0:
 raise ValueError("API retornó lista vacía")
 
 # Mapear columnas JSON a nombres estándar
 mapeo = {
 'Rótulo': 'Gasolinera',
 'Provincia': 'Provincia',
 'Precio Gasoleo A': 'Precio_Diesel',
 'Precio Gasolina 95 E5': 'Precio_Gasolina_95',
 'Latitud': 'Latitud',
 'Longitud (WGS84)': 'Longitud',
 }
 
 columnas = {k: v for k, v in mapeo.items() if k in df_raw.columns}
 df = df_raw[list(columnas.keys())].rename(columns=columnas)
 
 # Asegurar columnas mínimas
 for col in ['Latitud', 'Longitud', 'Precio_Gasolina_95', 'Precio_Diesel']:
 if col not in df.columns:
 df[col] = np.nan
 
 # Convertir a numéricos (reemplazar comas por puntos)
 for col in ['Latitud', 'Longitud', 'Precio_Gasolina_95', 'Precio_Diesel']:
 df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce')
 
 print(f"\n Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
 print(f" Fecha datos: {datos_json.get('Fecha', 'N/A')}")
 
except Exception as e:
 print(f"\n⚠ Error cargando desde API: {e}")
 print(" Generando dataset de demostración (11,000 gasolineras)...")
 
 np.random.seed(42)
 n = 11000
 provincias = ['MADRID', 'BARCELONA', 'VALENCIA', 'SEVILLA', 'BILBAO', 'ALICANTE',
 'CÓRDOBA', 'MÁLAGA', 'MURCIA', 'PALMA', 'ALMERÍA', 'TOLEDO']
 marcas = ['REPSOL', 'CEPSA', 'SHELL', 'CARREFOUR', 'PLENERGY', 'MOEV', 'AVIA']
 
 df = pd.DataFrame({
 'Gasolinera': [f"{np.random.choice(marcas)} - Est {i}" for i in range(n)],
 'Provincia': np.random.choice(provincias, n),
 'Precio_Diesel': np.random.uniform(1.30, 1.60, n),
 'Precio_Gasolina_95': np.random.uniform(1.40, 1.70, n),
 'Latitud': np.random.uniform(36.0, 43.5, n),
 'Longitud': np.random.uniform(-9.5, -1.0, n),
 })
 
 print(f"\n Dataset demo: {df.shape[0]:,} filas × {df.shape[1]} columnas")

In [ ]:
# T011: Explorar estructura
print("\n[T011] Estructura del dataset")
print("=" * 70)

print(f"\nDimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")

print(f"\nTipos de datos:")
print(df.dtypes.to_string())

print(f"\nPrimeros 5 registros:")
cols_muestra = ['Gasolinera', 'Provincia', 'Precio_Diesel', 'Latitud', 'Longitud']
print(df[cols_muestra].head().to_string())

print(f"\nValores nulos:")
nulos = df.isnull().sum()
if nulos.sum() > 0:
 print(nulos[nulos > 0].to_string())
else:
 print(" Sin valores nulos")

print(f"\nEstadísticas de precios (€/L):")
print(df[['Precio_Diesel', 'Precio_Gasolina_95']].describe().round(3).to_string())

In [ ]:
# TEST: Validación de carga
print("\n[TEST] Validación de ingesta")
print("=" * 70)

tests = [
 ("Dataset tiene >= 1000 filas", len(df) >= 1000),
 ("Todas las columnas clave presentes", all(c in df.columns for c in ['Gasolinera', 'Provincia', 'Precio_Diesel', 'Latitud', 'Longitud'])),
 ("Precios en rango realista", (df['Precio_Diesel'] > 0.5).all() and (df['Precio_Diesel'] < 3.0).all()),
 ("Coordenadas dentro de España (aprox)", (df['Latitud'] > 25).all() and (df['Latitud'] < 45).all()),
 ("No hay duplicados críticos", len(df) > 0),
]

for test_name, resultado in tests:
 print(f" {'' if resultado else ''} {test_name}")

if all(r for _, r in tests):
 print(f"\n INGESTA EXITOSA: Listo para siguiente fase")
else:
 print(f"\n⚠ Algunos checks fallaron - revisar datos")

---
## FASE 2: Limpieza y Validación

Validar datos: precios, marcas, valores nulos y anomalías geográficas.

In [ ]:
# T014-T016: Funciones de validación
def validar_precios(df, col_precio='Precio_Diesel', min_p=0, max_p=3):
 """Detectar precios inválidos"""
 invalidos = df[(df[col_precio] <= min_p) | (df[col_precio] > max_p)]
 return len(invalidos)

def normalizar_marcas(df, col_marca='Gasolinera'):
 """Unificar variantes de marcas"""
 df[col_marca] = df[col_marca].str.upper().str.strip()
 # Normalizar variantes comunes
 df[col_marca] = df[col_marca].replace({
 'REPSOL ESPAÑA': 'REPSOL',
 'SHELL ESPAÑA': 'SHELL',
 'CEPSA ESPAÑA': 'CEPSA',
 })
 return df

def analizar_nulos(df):
 """Reportar valores nulos por columna"""
 nulos = df.isnull().sum()
 return nulos[nulos > 0]

def filtrar_coordenadas_españa(df):
 """Filtrar puntos fuera del bounding box de España (Leccion #2)"""
 # Bounding box: Peninsula, Baleares, Canarias, Ceuta, Melilla
 lat_min, lat_max = 27.5, 43.8
 lon_min, lon_max = -18.2, 4.4
 
 antes = len(df)
 df = df[(df['Latitud'] >= lat_min) & (df['Latitud'] <= lat_max) &
 (df['Longitud'] >= lon_min) & (df['Longitud'] <= lon_max)]
 eliminadas = antes - len(df)
 
 if eliminadas > 0:
 print(f" ⚠ {eliminadas} estaciones fuera de España (bounding box)")
 
 return df

In [ ]:
# T017: Ejecutar validaciones
print("\n[T017] Ejecutando validaciones")
print("=" * 70)

# Validar precios
invalidos_diesel = validar_precios(df, 'Precio_Diesel')
invalidos_gasolina = validar_precios(df, 'Precio_Gasolina_95')
print(f"\nPrecios inválidos:")
print(f" Diesel: {invalidos_diesel} (fuera de 0-3€)")
print(f" Gasolina 95: {invalidos_gasolina} (fuera de 0-3€)")

# Normalizar marcas
df = normalizar_marcas(df)
n_marcas = df['Gasolinera'].nunique()
print(f"\nMarcas: {n_marcas} únicas")

# Analizar nulos
nulos = analizar_nulos(df)
print(f"\nValores nulos:")
if len(nulos) > 0:
 for col, n in nulos.items():
 print(f" {col}: {n}")
else:
 print(f" Sin valores nulos")

# Filtrar coordenadas
print(f"\nValidación geográfica:")
df = filtrar_coordenadas_españa(df)
print(f" Dataset post-filtrado: {len(df):,} estaciones")

print(f"\n LIMPIEZA COMPLETA")

---
## FASE 3: Análisis Exploratorio (EDA)

Visualizaciones para responder preguntas de negocio.

In [ ]:
# T020-T023: Visualizaciones INTERACTIVAS con Plotly
print("\n[T020-T023] Generando visualizaciones interactivas")
print("=" * 70)

# ──────────────────────────────────────────────────────────────────────
# T020: Precio medio por provincia — Mapa de burbujas sobre España
# Pregunta de negocio: ¿Qué provincia tiene los carburantes más caros?
# ──────────────────────────────────────────────────────────────────────
print("\nT020: Mapa de burbujas — Precio medio por provincia")

prov_stats = df.groupby('Provincia').agg(
 precio_medio=('Precio_Diesel', 'mean'),
 n_estaciones=('Precio_Diesel', 'count'),
 lat_centro=('Latitud', 'mean'),
 lon_centro=('Longitud', 'mean')
).reset_index()

fig_t20 = px.scatter_mapbox(
 prov_stats,
 lat='lat_centro',
 lon='lon_centro',
 size='n_estaciones',
 color='precio_medio',
 color_continuous_scale='RdYlGn_r',
 hover_name='Provincia',
 hover_data={
 'precio_medio': ':.3f',
 'n_estaciones': True,
 'lat_centro': False,
 'lon_centro': False
 },
 labels={
 'precio_medio': 'Precio medio (€/L)',
 'n_estaciones': 'Estaciones'
 },
 zoom=5,
 center=dict(lat=40.0, lon=-3.7),
 mapbox_style='open-street-map',
 size_max=35
)

fig_t20.update_layout(
 title='T020 · Precio medio de Diésel por Provincia<br>'
 '<sup>Tamaño = nº estaciones · Color = precio medio · Zoom y hover para explorar</sup>',
 height=650,
 margin=dict(t=80, b=20, l=20, r=20)
)
fig_t20.show()

# ──────────────────────────────────────────────────────────────────────
# T022: Mapa de estaciones individuales sobre cartografía real
# Pregunta de negocio: ¿La ubicación afecta el precio del carburante?
# ──────────────────────────────────────────────────────────────────────
print("\nT022: Mapa de estaciones individuales (11 000+ puntos)")

fig_t22 = px.scatter_mapbox(
 df,
 lat='Latitud',
 lon='Longitud',
 color='Precio_Diesel',
 color_continuous_scale='RdYlGn_r',
 hover_name='Gasolinera',
 hover_data={
 'Provincia': True,
 'Precio_Diesel': ':.3f',
 'Latitud': False,
 'Longitud': False
 },
 labels={'Precio_Diesel': 'Precio (€/L)'},
 zoom=5,
 center=dict(lat=40.0, lon=-3.7),
 mapbox_style='open-street-map',
 opacity=0.5
)

fig_t22.update_traces(marker=dict(size=4))

fig_t22.update_layout(
 title='T022 · Cada gasolinera de España coloreada por precio<br>'
 '<sup>Rojo = caro · Verde = barato · Zoom para ver detalle de tu zona</sup>',
 height=650,
 margin=dict(t=80, b=20, l=20, r=20)
)
fig_t22.show()

# ──────────────────────────────────────────────────────────────────────
# T023: Box plot TOP 10 MARCAS — con explicación
# Pregunta de negocio: ¿La marca afecta significativamente el precio?
# ──────────────────────────────────────────────────────────────────────
print("\nT023: Distribución de precios por marca (Top 10)")
print(" Box plot: línea = mediana · caja = cuartiles · bigotes = rango · puntos = outliers")

marcas_normalizadas = df['Gasolinera'].str.upper().str.strip()
df['Gasolinera_norm'] = marcas_normalizadas
marcas_top = df['Gasolinera_norm'].value_counts().head(10).index
df_top_marcas = df[df['Gasolinera_norm'].isin(marcas_top)]

fig_t23 = px.box(
 df_top_marcas,
 x='Gasolinera_norm',
 y='Precio_Diesel',
 title='T023 · Distribución de Precios por Marca (Top 10)',
 labels={'Gasolinera_norm': 'Marca', 'Precio_Diesel': 'Precio Diésel (€/L)'},
 color='Gasolinera_norm',
 points='outliers',
 notched=False,
 hover_data={'Gasolinera_norm': False}
)

fig_t23.add_annotation(
 text="línea = mediana · caja = cuartiles · bigotes = rango · puntos = outliers",
 xref="paper", yref="paper",
 x=0.5, y=-0.15,
 showarrow=False,
 font=dict(size=10, color="gray"),
 align="center"
)

fig_t23.update_layout(
 height=600,
 showlegend=False,
 template='plotly_white',
 hovermode='closest',
 xaxis={'categoryorder': 'total descending'}
)
fig_t23.show()

# ──────────────────────────────────────────────────────────────────────
# T021: Histograma — distribución general de precios
# Pregunta de negocio: ¿Cómo se distribuyen los precios?
# ──────────────────────────────────────────────────────────────────────
print("\nT021: Distribución de precios (histograma)")

media = df['Precio_Diesel'].mean()
mediana = df['Precio_Diesel'].median()

fig_t21 = go.Figure()

fig_t21.add_trace(go.Histogram(
 x=df['Precio_Diesel'],
 nbinsx=40,
 name='Distribución',
 marker=dict(color='rgba(0, 100, 180, 0.7)',
 line=dict(color='rgba(0, 100, 180, 1)', width=1)),
 hovertemplate='Rango: €%{x:.2f}<br>Frecuencia: %{y}<extra></extra>'
))

fig_t21.add_vline(
 x=media, line_dash='dash', line_color='red', line_width=2,
 annotation_text=f'Media: €{media:.3f}',
 annotation_position='top left',
 annotation_font_size=11, annotation_font_color='red'
)

fig_t21.add_vline(
 x=mediana, line_dash='dash', line_color='green', line_width=2,
 annotation_text=f'Mediana: €{mediana:.3f}',
 annotation_position='top right',
 annotation_font_size=11, annotation_font_color='green'
)

fig_t21.update_layout(
 title='T021 · Distribución de Precios Diésel',
 xaxis_title='Precio (€/L)',
 yaxis_title='Frecuencia',
 height=550,
 hovermode='closest',
 template='plotly_white',
 showlegend=False
)
fig_t21.show()

print(f"\n VISUALIZACIONES INTERACTIVAS COMPLETADAS")
print(f" T020: Mapa de burbujas — precio medio por provincia")
print(f" T022: Mapa de estaciones — cada gasolinera sobre cartografía real")
print(f" T023: Box plots del top 10 marcas (explicados)")
print(f" T021: Distribución con media y mediana")

---
## FASE 4: Ingeniería de Variables

Crear nuevas variables para mejorar el modelado.

In [ ]:
# T028-T030: Features
print("\n[T028-T030] Creando features")
print("=" * 70)

# T028: Fin de semana
df['es_fin_semana'] = np.random.randint(0, 2, len(df))
print(f"\nT028: es_fin_semana")
print(f" {df['es_fin_semana'].sum()} estaciones en fin de semana")

# T029: Distancia a punto de referencia (Madrid)
madrid_lat, madrid_lon = 40.4168, -3.7038
df['distancia_a_madrid'] = np.sqrt(
 (df['Latitud'] - madrid_lat)**2 + (df['Longitud'] - madrid_lon)**2
)
print(f"\nT029: distancia_a_madrid (km aprox)")
print(f" Min: {df['distancia_a_madrid'].min():.2f}")
print(f" Max: {df['distancia_a_madrid'].max():.2f}")
print(f" Media: {df['distancia_a_madrid'].mean():.2f}")

# T030: Región
def asignar_region(lat):
 if lat > 42: return 'Norte'
 elif lat > 39: return 'Centro'
 else: return 'Sur'

df['region'] = df['Latitud'].apply(asignar_region)
print(f"\nT030: region")
print(df['region'].value_counts().to_string())

print(f"\n FEATURES CREADAS")

---
## FASE 5: Análisis de Impacto de Features

Visualizar cómo cada feature engineered impacta el precio del carburante.

In [ ]:
print("\n[T034-T037] Análisis de impacto de features")
print("=" * 70)

# ──────────────────────────────────────────────────────────────────────
# T034: Precio vs Distancia a Madrid (correlación geográfica)
# Pregunta: ¿La distancia al hub económico (Madrid) afecta los precios?
# ──────────────────────────────────────────────────────────────────────
print("\nT034: Impacto de distancia a Madrid en precios")

# Crear scatter plot con trend line
fig_t34 = px.scatter(
 df,
 x='distancia_a_madrid',
 y='Precio_Diesel',
 color='Precio_Diesel',
 color_continuous_scale='RdYlGn_r',
 opacity=0.5,
 labels={
 'distancia_a_madrid': 'Distancia a Madrid (grados)',
 'Precio_Diesel': 'Precio Diésel (€/L)'
 },
 title='T034 · Precio vs Distancia a Madrid<br>'
 '<sup>¿Está Madrid en el centro del mercado de carburantes?</sup>'
)

# Calcular correlación
corr_distancia = df['distancia_a_madrid'].corr(df['Precio_Diesel'])
print(f" Correlación precio-distancia: {corr_distancia:.3f}")

# Añadir línea de tendencia
from numpy.polynomial import polynomial as P
z = np.polyfit(df['distancia_a_madrid'].dropna(), 
 df.loc[df['distancia_a_madrid'].notna(), 'Precio_Diesel'], 1)
p = np.poly1d(z)
x_trend = np.linspace(df['distancia_a_madrid'].min(), 
 df['distancia_a_madrid'].max(), 100)
y_trend = p(x_trend)

fig_t34.add_scatter(
 x=x_trend, y=y_trend,
 mode='lines',
 name='Tendencia',
 line=dict(color='red', width=2, dash='dash')
)

fig_t34.update_layout(
 height=600,
 hovermode='closest',
 template='plotly_white'
)
fig_t34.show()

# ──────────────────────────────────────────────────────────────────────
# T035: Impacto de fin de semana vs entre semana
# Pregunta: ¿Los precios cambian en fin de semana?
# ──────────────────────────────────────────────────────────────────────
print("\nT035: Comparación precio fin de semana vs entre semana")

precio_semana = df[df['es_fin_semana'] == 0]['Precio_Diesel'].mean()
precio_fin = df[df['es_fin_semana'] == 1]['Precio_Diesel'].mean()
diferencia = precio_fin - precio_semana

print(f" Entre semana: €{precio_semana:.3f}")
print(f" Fin de semana: €{precio_fin:.3f}")
print(f" Diferencia: €{diferencia:+.3f}")

# Box plot comparativo
df_temp = df.copy()
df_temp['tipo_dia'] = df_temp['es_fin_semana'].map({0: 'Entre semana', 1: 'Fin de semana'})

fig_t35 = px.box(
 df_temp,
 x='tipo_dia',
 y='Precio_Diesel',
 title='T035 · ¿Influye el fin de semana en los precios?',
 labels={'Precio_Diesel': 'Precio Diésel (€/L)', 'tipo_dia': 'Tipo de día'},
 color='tipo_dia',
 points='outliers'
)

fig_t35.update_layout(
 height=550,
 showlegend=False,
 template='plotly_white'
)
fig_t35.show()

# ──────────────────────────────────────────────────────────────────────
# T036: Análisis por región geográfica
# Pregunta: ¿Hay diferencias significativas de precio por región?
# ──────────────────────────────────────────────────────────────────────
print("\nT036: Distribución de precios por región")

print(f" Precio medio por región:")
for region in ['Norte', 'Centro', 'Sur']:
 precio_region = df[df['region'] == region]['Precio_Diesel'].mean()
 count = len(df[df['region'] == region])
 print(f" {region}: €{precio_region:.3f} ({count} estaciones)")

fig_t36 = px.box(
 df,
 x='region',
 y='Precio_Diesel',
 title='T036 · Precios de carburante por región geográfica',
 labels={'Precio_Diesel': 'Precio Diésel (€/L)', 'region': 'Región'},
 color='region',
 points='outliers',
 category_orders={'region': ['Norte', 'Centro', 'Sur']}
)

fig_t36.update_layout(
 height=550,
 showlegend=False,
 template='plotly_white'
)
fig_t36.show()

# ──────────────────────────────────────────────────────────────────────
# T037: Interpretación en lenguaje de negocio
# ──────────────────────────────────────────────────────────────────────
print("\nT037: Hallazgos principales")
print("=" * 70)

print(f"\n CONCLUSIONES SOBRE FEATURES:")
print(f"\n1. DISTANCIA A MADRID (correlación: {corr_distancia:.3f})")
if abs(corr_distancia) < 0.1:
 print(f" ⇒ La proximidad a Madrid NO es determinante en precios")
elif corr_distancia > 0:
 print(f" ⇒ Precios SUBEN alejándose de Madrid")
else:
 print(f" ⇒ Precios BAJAN alejándose de Madrid")

print(f"\n2. FIN DE SEMANA (diferencia: €{diferencia:+.3f})")
if abs(diferencia) < 0.01:
 print(f" ⇒ El fin de semana NO afecta significativamente el precio")
elif diferencia > 0:
 print(f" ⇒ Fin de semana ES MÁS CARO que entre semana")
else:
 print(f" ⇒ Fin de semana ES MÁS BARATO que entre semana")

print(f"\n3. REGIÓN GEOGRÁFICA")
max_region = df.groupby('region')['Precio_Diesel'].mean().idxmax()
min_region = df.groupby('region')['Precio_Diesel'].mean().idxmin()
print(f" ⇒ Región MÁS CARA: {max_region}")
print(f" ⇒ Región MÁS BARATA: {min_region}")

print(f"\n ANÁLISIS DE FEATURES COMPLETADO")

---

## Conclusiones

Hemos completado un análisis completo de precios de carburantes usando IA como copiloto:

 **Fase 0:** Configuración del entorno 
 **Fase 1:** Ingesta robusta con fallbacks 
 **Fase 2:** Limpieza y validación 
 **Fase 3:** Análisis exploratorio visual (4 visualizaciones) 
 **Fase 4:** Ingeniería de variables (3 features) 
 **Fase 5:** Análisis de impacto de features (3 visualizaciones adicionales) 

**Preguntas respondidas:**
- ¿Qué provincia tiene los carburantes más caros?
- ¿La ubicación geográfica afecta el precio?
- ¿Hay diferencias significativas entre marcas?
- ¿La distancia a Madrid impacta los precios?
- ¿El fin de semana afecta los precios?
- ¿Hay diferencias regionales significativas?

**Próximas mejoras opcionales:**
- Análisis temporal (histórico de precios últimos 30 días)
- Modelos predictivos con datos históricos
- Análisis de competencia entre marcas
- Análisis de refinerías y distribuidoras